# 13 - Derniers réglages sans réentraîner : symétries et nuages

Deux améliorations possibles du modèle retenu (fusion optique + radar, notebook 10), **sans
réentraîner** :

1. **Moyenne sur des symétries de l'image** (*test-time augmentation*) : le réseau analyse aussi
   l'image retournée (4 symétries) ou retournée et transposée (8), et on moyenne les
   probabilités. Coût à bord : 4 ou 8 passes au lieu d'une.
2. **Seuil de nuages avec le radar** : le système répond INCERTAIN dès que 30 % de l'image est
   nuageuse, même quand il a l'image radar, qui voit à travers les nuages. Peut-on relever ce
   seuil sans rater d'inondation ?

**Règles fixées avant de voir les résultats** (choix sur la validation uniquement ; le test ne
sert qu'à mesurer) :

- symétries : adoptées seulement si l'IoU de validation gagne au moins 0,005 **et** qu'aucune
  inondation de validation de plus n'est manquée ; parmi celles qui passent, la moins coûteuse ;
- seuil de nuages : le plus élevé parmi 0,3 / 0,4 / 0,5 / 0,7 / aucun, tel qu'en validation on
  ne manque **aucune inondation de plus** et que le taux de fausses alertes n'augmente pas de
  plus de 2 points par rapport à 0,3.

Pour chaque variante, le seuil d'alerte et le seuil d'ambiguïté sont recalculés avec le même
protocole que le modèle retenu (alerte : validation ; ambiguïté : 95e centile de l'entraînement).

Exécution : Colab, *Exécution > Tout exécuter*, **sans GPU** (environ 30 minutes).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "onnxruntime"], check=True)

import json, time
import numpy as np, pandas as pd
from src import donnees, alerte, embarque, decision, entrainement

noms = donnees.telecharger("data", radar=True)
images, labels, perms = [], [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    s1 = donnees.lire_radar(nom, "data")
    images.append(np.concatenate([s2, s1 * 100]).astype(np.int16))
    labels.append(label.astype(np.int8)); perms.append(perm)
df = pd.DataFrame({"image": noms, "partie": [donnees.partie(n) for n in noms]})
df["part_inondee"] = [donnees.part_inondee_vraie(l.astype(np.float32), p) for l, p in zip(labels, perms)]
df["alerte_vraie"] = df["part_inondee"] >= 0.05
idx = {p: df.index[df["partie"] == p].tolist() for p in ["train", "validation", "test"]}
sel = {p: (df["partie"] == p).to_numpy() for p in idx}
print({p: len(i) for p, i in idx.items()})

## 1. Probabilités d'eau avec 1, 4 et 8 symétries

In [ ]:
with open("one-look/modeles/fusion/config.json", encoding="utf-8") as f:
    cfg = json.load(f)
sess = embarque.session("one-look/modeles/fusion/" + cfg["modele"], fils=2)
detecteur = decision.DetecteurInconnu.depuis_config(cfg["detecteur_inconnu"])
print("Modèle :", cfg["modele"], "| seuils actuels :", cfg["seuils_decision"])

N = [1, 4, 8]
probs = {n: {} for n in N}
debut = time.time()
for k in range(len(noms)):
    x = images[k].astype(np.float32)
    une = [r(embarque.probabilite_eau(sess, np.ascontiguousarray(a(x)))) for a, r in embarque.transformations(8)]
    for n in N:
        probs[n][k] = np.mean(une[:n], axis=0).astype(np.float16)
    if k % 100 == 0:
        print(f"{k}/{len(noms)} images, {time.time() - debut:.0f} s")
print(f"terminé en {time.time() - debut:.0f} s")

## 2. Symétries : qualité de la carte et des alertes

In [ ]:
x = images[idx["validation"][0]].astype(np.float32)
temps = {n: embarque.latence_ms(sess, canaux=15) * n for n in N}   # n passes du réseau

nuages = np.array([decision.part_nuages(images[k][:13].astype(np.float32)) for k in range(len(noms))])
inconnu = np.array([detecteur.score(images[k].astype(np.float32)) for k in range(len(noms))])

def evaluer(n, seuil_nuages=0.30):
    P = probs[n]
    eau = np.array([alerte.part_eau_reseau(P[k].astype(np.float32), images[k][:13].astype(np.float32), perms[k]) for k in range(len(noms))])
    amb = np.array([decision.ambiguite(P[k], images[k][:13].astype(np.float32), perms[k]) for k in range(len(noms))])
    t_alerte, _ = alerte.choisir_seuil_alerte(df.loc[sel["validation"], "alerte_vraie"], eau[sel["validation"]], 0.05)
    seuils = {"alerte": t_alerte, "nuages": seuil_nuages,
              "ambiguite": round(float(np.quantile(amb[sel["train"]], 0.95)), 4), "inconnu": cfg["seuils_decision"]["inconnu"]}
    dec = np.array([decision.decider(e, c, a, i, seuils) for e, c, a, i in zip(eau, nuages, amb, inconnu)])
    ligne = {"symetries": n, "seuil_nuages": "aucun" if seuil_nuages > 1 else seuil_nuages,
             "t_alerte": t_alerte, "ambiguite": seuils["ambiguite"], "ms_1_coeur": round(temps[n])}
    for part in ["validation", "test"]:
        b = decision.bilan(dec[sel[part]], df.loc[sel[part], "alerte_vraie"])
        ligne.update({f"iou_{part}": round(entrainement.iou_eau(P, labels, idx[part]), 4),
                      f"incertain_{part}": b["INCERTAIN"], f"f1_{part}": b["f1"],
                      f"fausses_alertes_{part}": b["taux_fausse_alerte"],
                      f"manquees_{part}": b["inondations_manquees_sans_prevenir"]})
    return ligne, seuils

lignes = [evaluer(n)[0] for n in N]
sym = pd.DataFrame(lignes).set_index("symetries")
os.makedirs("resultats", exist_ok=True); sym.to_csv("resultats/symetries_13.csv")
pd.set_option("display.width", 250)
print(sym.to_string())

ref = sym.loc[1]
ok = [n for n in N[1:] if sym.loc[n, "iou_validation"] >= ref["iou_validation"] + 0.005
      and sym.loc[n, "manquees_validation"] <= ref["manquees_validation"]]
N_RETENU = min(ok) if ok else 1
print("\nSymétries retenues (règle sur la validation) :", N_RETENU)

## 3. Seuil de nuages quand le radar est disponible

In [ ]:
lignes = []
for t in [0.30, 0.40, 0.50, 0.70, 1.01]:
    lignes.append(evaluer(N_RETENU, t)[0])
nua = pd.DataFrame(lignes).set_index("seuil_nuages")
nua.to_csv("resultats/seuil_nuages_13.csv")
print(nua[[c for c in nua.columns if c not in ("symetries", "ms_1_coeur")]].to_string())

base = nua.iloc[0]
candidats = [s for s in nua.index
             if nua.loc[s, "manquees_validation"] <= base["manquees_validation"]
             and nua.loc[s, "fausses_alertes_validation"] <= base["fausses_alertes_validation"] + 0.02]
NUAGES_RETENU = candidats[-1]
print("\nSeuil de nuages retenu (règle sur la validation) :", NUAGES_RETENU)

## 4. Bilan et configuration proposée

In [ ]:
final, seuils_final = evaluer(N_RETENU, 1.01 if NUAGES_RETENU == "aucun" else float(NUAGES_RETENU))
avant = sym.loc[1]
comparaison = pd.DataFrame({"modèle retenu (notebook 10)": avant, "après réglages": pd.Series(final)})
print(comparaison.to_string())
comparaison.to_csv("resultats/bilan_13.csv")
proposition = {"symetries": N_RETENU, "seuils_decision": seuils_final}
print("\nConfiguration proposée :", json.dumps(proposition))
with open("resultats/config_13.json", "w") as f:
    json.dump(proposition, f, indent=1)

import shutil
shutil.make_archive("resultats_13", "zip", "resultats")
try:
    from google.colab import files
    files.download("resultats_13.zip")
except ImportError:
    print("Hors de Colab : resultats_13.zip")